## NTU decoders: block error probability → LER per round

Start Jupyter from the directory containing this notebook, then select **Restart Kernel and Run All Cells**. Dependencies: `pandas`, `numpy`, `scipy`, and `ipykernel`.

The CSV column `ler` records the block-level logical error probability over the full $R$-round memory experiment, rather than the logical error rate per round:

$$P_{\mathrm{block}}(R)=1-\mathrm{block\ accuracy}.$$

Convert this probability using the supplied formula from NTU decoder paper:

$$\epsilon_{\mathrm{round}}=\frac{1}{2}\left[1-\left(1-2P_{\mathrm{block}}(R)\right)^{1/R}\right].$$

Each row represents an independent run. Preserve the original CSV column `ler`, copy it to the explicitly named `P_block` column, and use the corresponding row's `rounds` value to calculate `LER_per_round`. This conversion assumes the identical, independent per-round logical-flip model underlying the supplied formula. No additional normalization by the number of logical qubits is applied, and `per_log_mean` / `per_log_worst` are not used. The inversion uses the branch $0\le P_{\mathrm{block}}\le 0.5$.


### 1. Read the original `ler` values for each run

In [1]:
from pathlib import Path
from IPython.display import display
import numpy as np
import pandas as pd

DATA_DIR = Path("datasets/NTU_decoders")
RESULT_FILES = {
    "NTU-Neural-BP": DATA_DIR / "bb72_neuralbp_eval_r6_p0.005.csv",
    "NTU-Transformer": DATA_DIR / "bb72_transformer_eval_r6_p0.005.csv",
}
REQUIRED_COLUMNS = {
    "block_size", "rounds", "p", "total_samples", "correct", "errors", "block_acc", "ler"
}

frames = []
for decoder, path in RESULT_FILES.items():
    frame = pd.read_csv(path)
    missing = REQUIRED_COLUMNS - set(frame.columns)
    if missing:
        raise ValueError(f"{path}: missing columns {sorted(missing)}")
    for column in sorted(REQUIRED_COLUMNS):
        frame[column] = pd.to_numeric(frame[column], errors="raise")
    if frame.empty or not np.isfinite(frame[sorted(REQUIRED_COLUMNS)].to_numpy()).all():
        raise ValueError(f"{path}: empty input or non-finite values")
    frame.insert(0, "decoder", decoder)
    frame.insert(1, "run", np.arange(1, len(frame) + 1))
    frame.insert(2, "source_file", path.as_posix())
    frames.append(frame)

raw_results = pd.concat(frames, ignore_index=True)
display(raw_results[["decoder", "run", "rounds", "block_acc", "ler", "errors", "total_samples"]])


,decoder,run,rounds,block_acc,ler,errors,total_samples
0,NTU-Neural-BP,1,6,0.8610,0.1390,139,1000
1,NTU-Neural-BP,2,6,0.8620,0.1380,138,1000
2,NTU-Neural-BP,3,6,0.8570,0.1430,143,1000
3,NTU-Neural-BP,4,6,0.8550,0.1450,145,1000
4,NTU-Neural-BP,5,6,0.8480,0.1520,152,1000
5,NTU-Neural-BP,6,6,0.8610,0.1390,139,1000
6,NTU-Transformer,1,6,0.9199,0.0801,801,10000
7,NTU-Transformer,2,6,0.9184,0.0816,816,10000
8,NTU-Transformer,3,6,0.9162,0.0838,838,10000
9,NTU-Transformer,4,6,0.9233,0.0767,767,10000


### 2. Validate block probabilities and convert each run

First verify that `ler = 1 - block_acc = errors / total_samples`. The numerical tolerance allows for decimal rounding in the CSV files. The calculation uses `log1p` and `expm1` for numerical stability at small probabilities and is equivalent to the formula above.

In [2]:
def block_probability_to_ler_per_round(p_block, rounds):
    """Invert P_block = [1 - (1 - 2 * epsilon_round)**R] / 2."""
    probability, round_count = np.broadcast_arrays(
        np.asarray(p_block, dtype=float), np.asarray(rounds, dtype=float)
    )
    if not np.all(np.isfinite(probability) & (probability >= 0) & (probability <= 0.5)):
        raise ValueError("P_block must be finite and in [0, 0.5].")
    if not np.all(np.isfinite(round_count) & (round_count > 0) & (round_count == np.floor(round_count))):
        raise ValueError("rounds must be positive finite integers.")
    # P_block = 0.5 gives log(0) = -inf and epsilon_round = 0.5.
    with np.errstate(divide="ignore"):
        return -0.5 * np.expm1(np.log1p(-2 * probability) / round_count)


if not raw_results["total_samples"].gt(0).all():
    raise ValueError("total_samples must be positive.")
if not np.allclose(raw_results["correct"] + raw_results["errors"], raw_results["total_samples"], rtol=0, atol=0):
    raise ValueError("correct + errors must equal total_samples.")
if not raw_results["block_acc"].between(0, 1).all():
    raise ValueError("block_acc must be in [0, 1].")
if not np.allclose(raw_results["ler"], 1 - raw_results["block_acc"], rtol=0, atol=1e-7):
    raise ValueError("CSV ler disagrees with 1 - block_acc.")
if not np.allclose(raw_results["ler"], raw_results["errors"] / raw_results["total_samples"], rtol=0, atol=1e-7):
    raise ValueError("CSV ler disagrees with errors / total_samples.")

results = raw_results.copy()
results["P_block"] = results["ler"]  # Preserve the original CSV values.
results["LER_per_round"] = block_probability_to_ler_per_round(
    results["P_block"], results["rounds"]
)

# Check boundaries, R=1, and reconstruct each original block probability.
np.testing.assert_allclose(block_probability_to_ler_per_round([0, 0.2, 0.5], 1), [0, 0.2, 0.5])
np.testing.assert_allclose(block_probability_to_ler_per_round([0, 0.5], 6), [0, 0.5])
reconstructed = 0.5 * (1 - (1 - 2 * results["LER_per_round"]) ** results["rounds"])
np.testing.assert_allclose(reconstructed, results["P_block"], rtol=1e-12, atol=1e-14)

with pd.option_context("display.float_format", "{:.8f}".format):
    display(results[["decoder", "run", "block_size", "rounds", "p", "ler", "P_block", "LER_per_round"]])


,decoder,run,block_size,rounds,p,ler,P_block,LER_per_round
0,NTU-Neural-BP,1,72,6,0.00500000,0.13900000,0.13900000,0.02642053
1,NTU-Neural-BP,2,72,6,0.00500000,0.13800000,0.13800000,0.02620214
2,NTU-Neural-BP,3,72,6,0.00500000,0.14300000,0.14300000,0.02729916
3,NTU-Neural-BP,4,72,6,0.00500000,0.14500000,0.14500000,0.02774156
4,NTU-Neural-BP,5,72,6,0.00500000,0.15200000,0.15200000,0.02930649
5,NTU-Neural-BP,6,72,6,0.00500000,0.13900000,0.13900000,0.02642053
6,NTU-Transformer,1,72,6,0.00500000,0.08010000,0.08010000,0.01433965
7,NTU-Transformer,2,72,6,0.00500000,0.08160000,0.08160000,0.01462923
8,NTU-Transformer,3,72,6,0.00500000,0.08380000,0.08380000,0.01505552
9,NTU-Transformer,4,72,6,0.00500000,0.07670000,0.07670000,0.01368644


### 3. Summarize independent runs with 95% confidence intervals

Convert each run first, then summarize runs with the same decoder, block size, rounds, and physical error probability `p`. Retain the sample standard deviation (`ddof=1`) and calculate a two-sided confidence interval for each mean:

$$\bar{x} \pm t_{0.975,\,n-1}\frac{s}{\sqrt{n}}.$$

Here, $n$ is the number of independent runs and $s$ is their sample standard deviation. These intervals estimate the mean across runs, assuming independent, approximately normally distributed run-level estimates. They are not pooled-shot binomial confidence intervals. For LER per round, compute the interval from the individually converted runs, rather than converting an interval for the mean block probability. At least two runs are required; otherwise interval bounds are `NaN`. Bounds are left unmodified rather than clipped to the probability range.


In [3]:
from scipy.stats import t

summary = (
    results.groupby(["decoder", "block_size", "rounds", "p"], sort=False, as_index=False)
    .agg(
        runs=("run", "count"),
        
        block_acc_mean=("block_acc", "mean"),
        block_acc_std=("block_acc", "std"),
        P_block_mean=("P_block", "mean"),
        P_block_std=("P_block", "std"),
        LER_per_round_mean=("LER_per_round", "mean"),
        LER_per_round_std=("LER_per_round", "std"),
    )
)
# Two-sided 95% confidence intervals for the mean across independent runs.
critical_t = t.ppf(0.975, df=summary["runs"] - 1)
for metric in ["block_acc", "P_block", "LER_per_round"]:
    margin = critical_t * summary[f"{metric}_std"] / np.sqrt(summary["runs"])
    summary[f"{metric}_ci95_low"] = summary[f"{metric}_mean"] - margin
    summary[f"{metric}_ci95_high"] = summary[f"{metric}_mean"] + margin

# Group each mean, SD, and interval together for easier comparison.
summary = summary[
    ["decoder", "block_size", "rounds", "p", "runs"]
    + [f"{metric}_{suffix}"
       for metric in ["block_acc", "P_block", "LER_per_round"]
       for suffix in ["mean", "std", "ci95_low", "ci95_high"]]
]
with pd.option_context("display.float_format", "{:.8f}".format):
    display(summary)


,decoder,block_size,rounds,p,runs,block_acc_mean,block_acc_std,block_acc_ci95_low,block_acc_ci95_high,P_block_mean,P_block_std,P_block_ci95_low,P_block_ci95_high,LER_per_round_mean,LER_per_round_std,LER_per_round_ci95_low,LER_per_round_ci95_high
0,NTU-Neural-BP,72,6,0.00500000,6,0.85733333,0.00531664,0.85175386,0.86291281,0.14266667,0.00531664,0.13708719,0.14824614,0.02723173,0.00117858,0.02599489,0.02846858
1,NTU-Transformer,72,6,0.00500000,6,0.91913333,0.00262196,0.91638176,0.92188491,0.08086667,0.00262196,0.07811509,0.08361824,0.01448865,0.00050571,0.01395794,0.01501936


### 4. Surface-code d=7 results

Analyze `datasets/NTU_decoders/eval_d7_results.csv` separately because its schema and code differ from the BB72 results above. Each row is treated as an independent evaluation run. The CSV supplies distance `d`, shot counts, block probability `ler`, and `ler_per_round`; it does not supply `rounds` or physical error probability `p`.

The round count is established by the source code in `../benchmark/surface code/NTU-Transformer/`:

- `transformer.py` declares `--rounds` with `default=-1`, then replaces `-1` with `args.d` before training. An explicit `--rounds` can override this training default.
- `OnlineSurfaceCodeDataset` in `transformer.py` passes its `rounds` argument directly to `stim.Circuit.generated(..., rounds=rounds, distance=d)`.
- The evaluation script `inference.py` constructs `FullMapper(d, d)` and `OnlineSurfaceCodeDataset(d, d, eval_p, bs, is_eval=True)`. Its evaluation circuit therefore uses **`R = d` rounds**, independently of the configurable training default. For d7 results generated by this script, **`R = 7`**. The `rounds` column below is derived from this evaluation implementation, rather than recorded in the CSV.

The evaluation script defines the CSV's `ler_per_round` as `ler / d`, which here equals `P_block / d`. Preserve and summarize that original linear normalization as well as `P_block = ler`. This transformation can be found in the Evaluation metrics of NTU paper: 
$$
\varepsilon_{LQ} \approx \frac{P_{\mathrm{block}}(R)}{kR},
$$
where \(k\) denotes the number of logical qubits (\(k=1\)).

In [4]:
d7_path = DATA_DIR / "eval_d7_results.csv"
d7_results = pd.read_csv(d7_path)
d7_required = {"d", "total_samples", "total_errors", "ler", "ler_per_round"}
missing = d7_required - set(d7_results.columns)
if missing:
    raise ValueError(f"{d7_path}: missing columns {sorted(missing)}")
for column in sorted(d7_required):
    d7_results[column] = pd.to_numeric(d7_results[column], errors="raise")
if d7_results.empty or not np.isfinite(d7_results[sorted(d7_required)].to_numpy()).all():
    raise ValueError(f"{d7_path}: empty input or non-finite values")
for column in ["d", "total_samples", "total_errors"]:
    values = d7_results[column]
    if not (values == np.floor(values)).all():
        raise ValueError(f"{column} must contain integers.")
if not d7_results["d"].eq(7).all():
    raise ValueError("Expected distance d=7 in eval_d7_results.csv.")
if not d7_results["total_samples"].gt(0).all():
    raise ValueError("total_samples must be positive.")
if not d7_results["total_errors"].between(0, d7_results["total_samples"]).all():
    raise ValueError("total_errors must be between zero and total_samples.")
if not np.allclose(d7_results["ler"], d7_results["total_errors"] / d7_results["total_samples"], rtol=0, atol=1e-7):
    raise ValueError("CSV ler disagrees with total_errors / total_samples.")
if not np.allclose(d7_results["ler_per_round"], d7_results["ler"] / d7_results["d"], rtol=0, atol=1e-7):
    raise ValueError("CSV ler_per_round disagrees with ler / d.")

d7_results.insert(0, "run", np.arange(1, len(d7_results) + 1))
# inference.py evaluates OnlineSurfaceCodeDataset(d, d, ...): R = d.
d7_results["rounds"] = d7_results["d"]
d7_results["accuracy"] = 1 - d7_results["ler"]
d7_results["P_block"] = d7_results["ler"]
# d7_results["LER_per_round"] = block_probability_to_ler_per_round(
#     d7_results["P_block"], d7_results["rounds"]
# )
with pd.option_context("display.float_format", "{:.8f}".format):
    display(d7_results)


,run,d,total_samples,total_errors,ler,ler_per_round,rounds,accuracy,P_block
0,1,7,10000,49,0.00490000,0.00070000,7,0.99510000,0.00490000
1,2,7,10000,41,0.00410000,0.00058571,7,0.99590000,0.00410000
2,3,7,10000,34,0.00340000,0.00048571,7,0.99660000,0.00340000
3,4,7,10000,42,0.00420000,0.00060000,7,0.99580000,0.00420000
4,5,7,10000,56,0.00560000,0.00080000,7,0.99440000,0.00560000
5,6,7,10000,47,0.00470000,0.00067143,7,0.99530000,0.00470000


In [5]:
d7_metrics = ["accuracy", "ler", "P_block", "ler_per_round"]
d7_summary = (
    d7_results.groupby(["d", "rounds"], as_index=False)
    .agg(
        runs=("run", "count"),
        total_samples=("total_samples", "sum"),
        total_errors=("total_errors", "sum"),
        **{f"{metric}_{stat}": (metric, stat)
           for metric in d7_metrics for stat in ["mean", "std"]},
    )
)
d7_critical_t = t.ppf(0.975, df=d7_summary["runs"] - 1)
for metric in d7_metrics:
    margin = d7_critical_t * d7_summary[f"{metric}_std"] / np.sqrt(d7_summary["runs"])
    d7_summary[f"{metric}_ci95_low"] = d7_summary[f"{metric}_mean"] - margin
    d7_summary[f"{metric}_ci95_high"] = d7_summary[f"{metric}_mean"] + margin
d7_summary = d7_summary[
    ["d", "rounds", "runs", "total_samples", "total_errors"]
    + [f"{metric}_{suffix}" for metric in d7_metrics
       for suffix in ["mean", "std", "ci95_low", "ci95_high"]]
]
with pd.option_context("display.float_format", "{:.8f}".format):
    display(d7_summary)


,d,rounds,runs,total_samples,total_errors,accuracy_mean,accuracy_std,accuracy_ci95_low,accuracy_ci95_high,ler_mean,...,ler_ci95_low,ler_ci95_high,P_block_mean,P_block_std,P_block_ci95_low,P_block_ci95_high,ler_per_round_mean,ler_per_round_std,ler_per_round_ci95_low,ler_per_round_ci95_high
0,7,7,6,60000,269,0.99551667,0.00075741,0.99472182,0.99631152,0.00448333,...,0.00368848,0.00527818,0.00448333,0.00075741,0.00368848,0.00527818,0.00064048,0.00010820,0.00052692,0.00075403


### 5. d7 means, sample standard deviations, and 95% confidence intervals

`accuracy = 1 - ler` is the block-level accuracy. Summarize the numeric evaluation fields using sample standard deviations (`ddof=1`). For `accuracy`, `ler` (block LER), `P_block` (the same block probability), CSV `ler_per_round` (linear normalization), and converted `LER_per_round` (nonlinear inversion), report two-sided 95% confidence intervals for the mean across independent runs, following Section 3:

$$\bar{x} \pm t_{0.975,n-1} s / \sqrt{n}.$$

These intervals describe uncertainty in the mean across runs, rather than pooled-shot binomial uncertainty. Per-round intervals are computed from the corresponding run-level values. Fewer than two runs yield undefined (`NaN`) intervals; bounds are not clipped.


In [6]:
d7_mean_std_columns = [
    "d", "total_samples", "total_errors", "accuracy", "ler", "ler_per_round",
    "rounds", "P_block"
]
d7_mean_std = d7_results[d7_mean_std_columns].agg(["mean", "std"])

d7_confidence_intervals = pd.DataFrame(index=pd.Index(d7_metrics, name="metric"))
for metric in d7_metrics:
    values = d7_results[metric]
    count = values.count()
    mean = values.mean()
    std = values.std(ddof=1)
    margin = t.ppf(0.975, df=count - 1) * std / np.sqrt(count)
    d7_confidence_intervals.loc[metric, ["mean", "std", "ci95_low", "ci95_high"]] = [
        mean, std, mean - margin, mean + margin
    ]
with pd.option_context("display.float_format", "{:.8f}".format):
    display(d7_mean_std)
    display(d7_confidence_intervals)


,d,total_samples,total_errors,accuracy,ler,ler_per_round,rounds,P_block
mean,7.00000000,10000.00000000,44.83333333,0.99551667,0.00448333,0.00064048,7.00000000,0.00448333
std,0.00000000,0.00000000,7.57407860,0.00075741,0.00075741,0.00010820,0.00000000,0.00075741


,mean,std,ci95_low,ci95_high
metric,,,,
accuracy,0.99551667,0.00075741,0.99472182,0.99631152
ler,0.00448333,0.00075741,0.00368848,0.00527818
P_block,0.00448333,0.00075741,0.00368848,0.00527818
ler_per_round,0.00064048,0.00010820,0.00052692,0.00075403
